# AI Image Generation Chatbot
Task: Generate images from text prompts using Azure OpenAI (`gpt-4o`).

Credentials are loaded securely from `.env`.

In [ ]:
import os
from dotenv import load_dotenv
import requests
import urllib.parse
import io
from PIL import Image
from IPython.display import display

# Load environment variables from .env
load_dotenv()

API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "your_azure_api_key_here")
ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "https://image-chatbot.services.ai.azure.com/openai/v1/responses")
MODEL = os.getenv("AZURE_OPENAI_MODEL", "gpt-4o")

### Test Connection

In [ ]:
headers = {
    "api-key": API_KEY,
    "Content-Type": "application/json"
}

body = {
    "model": MODEL,
    "input": "Hello, confirm connection."
}

res = requests.post(ENDPOINT, headers=headers, json=body, timeout=15)
print("Status:", res.status_code)
if res.status_code == 200:
    print("Azure OpenAI connection verified successfully.")

### Image Generation Function

In [ ]:
def generate_image(prompt_text, style="photorealistic"):
    print(f"Prompt: '{prompt_text}'")
    
    headers = {"api-key": API_KEY, "Content-Type": "application/json"}
    body = {
        "model": MODEL,
        "input": f"Turn this into a 20-word visual description for image generation in {style} style: {prompt_text}"
    }
    
    final_prompt = prompt_text
    try:
        r = requests.post(ENDPOINT, headers=headers, json=body, timeout=10)
        if r.status_code == 200:
            data = r.json()
            for item in data.get("output", []):
                for c in item.get("content", []):
                    if c.get("type") == "output_text":
                        final_prompt = c.get("text", "").strip('"\n ')
    except Exception:
        pass
    
    print("Enhanced prompt:", final_prompt)
    
    url = f"https://image.pollinations.ai/prompt/{urllib.parse.quote(final_prompt)}"
    web_headers = {
        "Referer": "https://pollinations.ai/",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    
    img_res = requests.get(url, headers=web_headers, timeout=35)
    if img_res.status_code == 200:
        img = Image.open(io.BytesIO(img_res.content))
        display(img)
        return img
    else:
        print("Error loading image. Status:", img_res.status_code)
        return None

### Test Prompt

In [ ]:
# Test prompt
car = generate_image("mercedes benz s class luxury car")

### Launch Streamlit App

In [ ]:
!streamlit run app.py